# Miro Face Detector - CNN Model Training

This notebook trains a Convolutional Neural Network (CNN) to classify images as either 'Miro' (1) or 'Random' (0). The data should be preprocessed by `src/build_dataset.py` before running this notebook. It is designed to be executable in Google Colab as requested.

## Configuration

Ensure your `config.yaml` file is properly positioned before training. For example, if the drive mounted fine and the location of the dataset is at `/content/drive/MyDrive/processed`, place the config file there or at the project root.

Example `config.yaml` content:
```yaml
model:
  img_size: 128
  output_path: /content/drive/MyDrive/processed/models/miro_detector.h5
data:
  dataset_csv: /content/drive/MyDrive/processed/dataset.csv
```

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
yaml_path = '../config.yaml'
if os.path.exists(yaml_path):
    import yaml
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    DATA_CSV = '../' + config['data']['dataset_csv']
    IMG_SIZE = config['model']['img_size']
    MODEL_OUTPUT = '../' + config['model']['output_path']
    DATA_DIR = '../'
else:
    # Fallback for Colab execution where structure might differ
    DATA_CSV = '/content/drive/MyDrive/processed/dataset.csv'
    IMG_SIZE = 128
    MODEL_OUTPUT = 'models/miro_detector.h5'
    DATA_DIR = '/content/drive/MyDrive/processed/'

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


## 1. Data Loading and Preparation

In [8]:
df = pd.read_csv(DATA_CSV)

def load_images_from_df(dataframe):
    images = []
    labels = []
    for idx, row in dataframe.iterrows():
        # Handle Windows paths in Colab
        filepath = str(row['filepath']).replace(chr(92), '/')
        img_path = os.path.join(DATA_DIR, filepath)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(row['label'])
        else:
            print(f"Error loading image: {img_path}")
    return np.array(images) / 255.0, np.array(labels)

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

print("Loading training data...")
X_train, y_train = load_images_from_df(train_df)
print("Loading testing data...")
X_test, y_test = load_images_from_df(test_df)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

Loading training data...
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0001.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0002.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0006.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0007.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0008.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0011.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0013.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0015.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_frame_0016.jpg
Error loading image: /content/drive/MyDrive/processed/data/processed/positive/video_

## 2. Model Architecture

Building a custom CNN following the assignment specification: Convolutional -> Max Pooling -> Flatten -> Dense.

In [9]:
model = models.Sequential([
    # Convolutional & Pooling Layer 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D((2, 2)),

    # Convolutional & Pooling Layer 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Convolutional & Pooling Layer 3
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flatten 2D feature maps to 1D vector
    layers.Flatten(),

    # Dense Fully Connected layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid') # Binary classification
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

## 3. Training and Evaluation

In [10]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test),
    batch_size=32
)

Epoch 1/10


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(32,), dtype=float32). Expected shape (None, 128, 128, 3), but input has incompatible shape (32,)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32,), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
plt.show()

## 4. Exporting Model Weights

In [ ]:
os.makedirs(os.path.dirname(MODEL_OUTPUT), exist_ok=True)
model.save(MODEL_OUTPUT)
print(f"Model saved to {MODEL_OUTPUT}")